In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize
from matplotlib import pyplot as plt
from tqdm import tqdm
from torch import nn


import torch
import numpy as np
import pandas as pd
import os

plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
housing = fetch_california_housing(data_home='../data')

print(housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [4]:
x_train_, x_test, y_train_, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_, y_train_, test_size=0.25, random_state=42
)

dataset_maps = {
    'train': (x_train, y_train),
    'val': (x_val, y_val),
    'test': (x_test, y_test)
}

In [ ]:
std = StandardScaler()
std.fit(x_train)


class HousingDataset(Dataset):
    def __init__(self, dataset_type='train'):
        self.x, self.y = dataset_maps[dataset_type]
        self.x = std.transform(self.x)
        self.x = torch.tensor(self.x, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
        # 多输入模型的输入格式
        # return (self.x[idx], self.x[idx][-2:]), self.y[idx]


train_ds = HousingDataset('train')
val_ds = HousingDataset('val')
test_ds = HousingDataset('test')

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

# 构建widedeep模型(单输入)
# 模型架构：Wide部分直接使用输入特征，Deep部分通过全连接层提取特征，最后将两部分特征拼接后进行预测。


class WideDeep(nn.Module):
    # input_dim=(8, 2) 多输入模型的输入维度
    # 取前8维作为Deep部分输入，后2维作为Wide部分输入
    def __init__(self, input_dim=8):
        super(WideDeep, self).__init__()
        # Deep部分通过全连接层提取特征
        self.deep = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        # 输出层
        self.output = nn.Linear(input_dim + 64, 1)

    # def forward(self, x_wide, x_deep) 多输入
    def forward(self, x):
        # Deep部分通过全连接层提取特征
        deep_out = self.deep(x)
        # 拼接Wide输入特征和Deep部分提取的特征进行预测
        concat = torch.cat([x, deep_out], dim=1)
        logits = self.output(concat)
        return logits

In [ ]:
std = StandardScaler()
std.fit(x_train)


class HousingDataset(Dataset):
    def __init__(self, dataset_type='train'):
        self.x, self.y = dataset_maps[dataset_type]
        self.x = std.transform(self.x)
        self.x = torch.tensor(self.x, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        # 多输入模型的输入格式
        return (self.x[idx], self.x[idx][-2:]), self.y[idx]


train_ds = HousingDataset('train')
val_ds = HousingDataset('val')
test_ds = HousingDataset('test')

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

# 构建widedeep模型（多输入）
# 模型架构：Wide部分直接使用输入特征，Deep部分通过全连接层提取特征，最后将两部分特征拼接后进行预测。


class WideDeep(nn.Module):
    # input_dim=(8, 2) 多输入模型的输入维度
    # 取前8维作为Deep部分输入，后2维作为Wide部分输入
    def __init__(self, input_dim=(8, 2)):
        super(WideDeep, self).__init__()
        # Deep部分通过全连接层提取特征
        self.deep = nn.Sequential(
            nn.Linear(input_dim[0], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        # 输出层
        self.output = nn.Linear(input_dim[1] + 64, 1)

    # 多输入 训练时 model(x_deep, x_wide)
    def forward(self, x_wide, x_deep):
        # Deep部分通过全连接层提取特征
        deep_out = self.deep(x_deep)
        # 拼接Wide输入特征和Deep部分提取的特征进行预测
        concat = torch.cat([x_wide, deep_out], dim=1)
        logits = self.output(concat)
        return logits